# EDA - raw source data

In [1]:
import duckdb

con = duckdb.connect()

customers = "read_csv_auto('../data/raw/customers.csv')"
orders = "read_csv_auto('../data/raw/orders.csv')"
order_items = "read_csv_auto('../data/raw/order_items.csv')"
products = "read_json_auto('../data/raw/products.json')"
fx_rates = "read_json_auto('../data/raw/fx_rates.json')"


## 1. Row counts

In [2]:
con.execute(f'''
    select 'customers' as source, count(*) as rows from {customers}
    union all select 'orders', count(*) from {orders}
    union all select 'order_items', count(*) from {order_items}
    union all select 'products', count(*) from {products}
''').df()


,source,rows
0,customers,50
1,orders,453
2,order_items,363
3,products,100


## 2. Key uniqueness (id)

In [3]:
con.execute(f'''
    select 'orders' as source, count(*) as rows, count(distinct id) as distinct_ids from {orders}
    union all
    select 'order_items', count(*), count(distinct id) from {order_items}
    union all
    select 'customers', count(*), count(distinct id) from {customers}
''').df()


,source,rows,distinct_ids
0,orders,453,453
1,order_items,363,363
2,customers,50,50


## 3. Orders with zero line items

In [4]:
con.execute(f'''
    select count(*) as orders_without_items
    from {orders} o
    left join {order_items} i on o.id = i.order_id
    where i.order_id is null
''').df()


,orders_without_items
0,93


## 4. order_items.product_id not in products.json

In [5]:
con.execute(f'''
    select i.product_id, count(*) as n
    from {order_items} i
    left join {products} p on i.product_id = p.id
    where p.id is null
    group by 1 order by 1
''').df()


,product_id,n
0,101,9
1,102,8
2,103,7
3,104,8
4,105,7
5,106,8
6,107,7
7,108,7
8,109,6
9,110,4


## 5. Overlap: invalid currency and orphan product_id, same item

In [6]:
con.execute(f'''
    select
        (i.currency not in ('USD','EUR','GBP')) as invalid_currency,
        (p.id is null) as orphan_product,
        count(*) as n
    from {order_items} i
    left join {products} p on i.product_id = p.id
    group by 1, 2 order by 1, 2
''').df()


,invalid_currency,orphan_product,n
0,False,False,285
1,False,True,3
2,True,False,7
3,True,True,68


## 6. orders.currency

In [7]:
con.execute(f"select currency, count(*) as n from {orders} group by 1 order by 2 desc").df()


,currency,n
0,USD,244
1,EUR,126
2,XYZ,27
3,ABC,27
4,QWE,25
5,GBP,4


## 7. order_items.currency

In [8]:
con.execute(f"select currency, count(*) as n from {order_items} group by 1 order by 2 desc").df()


,currency,n
0,USD,186
1,EUR,99
2,ABC,26
3,XYZ,25
4,QWE,24
5,GBP,3


## 8. orders.status

In [9]:
con.execute(f"select distinct status from {orders}").df()


,status
0,completed


## 9. Header currency vs item currency, per order

In [10]:
con.execute(f'''
    with per_order as (
        select o.id, o.currency as header_currency, min(i.currency) as item_currency
        from {orders} o
        join {order_items} i on o.id = i.order_id
        group by 1, 2
    )
    select
        count(*) as orders_with_items,
        sum(case when header_currency != item_currency then 1 else 0 end) as currency_mismatch
    from per_order
''').df()


,orders_with_items,currency_mismatch
0,360,114.0


## 10. orders.total_amount vs sum(order_items)

In [11]:
con.execute(f'''
    with items_rollup as (
        select order_id, sum(quantity * unit_price) as rollup_total
        from {order_items}
        group by 1
    )
    select
        count(*) as orders_with_items,
        sum(case when abs(o.total_amount - r.rollup_total) > 0.01 then 1 else 0 end) as amount_mismatch
    from {orders} o
    join items_rollup r on o.id = r.order_id
''').df()


,orders_with_items,amount_mismatch
0,360,193.0


## 11. total_amount mismatch, isolated from currency mismatch

In [12]:
con.execute(f'''
    with items_rollup as (
        select order_id, sum(quantity * unit_price) as rollup_total, min(currency) as item_currency
        from {order_items}
        group by 1
    ), joined as (
        select
            (abs(o.total_amount - r.rollup_total) > 0.01) as amount_mismatch,
            (o.currency != r.item_currency) as currency_mismatch
        from {orders} o
        join items_rollup r on o.id = r.order_id
    )
    select amount_mismatch, currency_mismatch, count(*) as n
    from joined group by 1, 2 order by 1, 2
''').df()


,amount_mismatch,currency_mismatch,n
0,False,False,167
1,True,False,79
2,True,True,114


## 12. order_date range vs fx_rates date coverage

In [13]:
order_range = con.execute(f"select min(order_date), max(order_date), count(distinct order_date::date) from {orders}").fetchone()
fx_dates = con.execute(f"select distinct r.date from (select unnest(responses) as r from {fx_rates}) order by 1").df()
print('order_date min/max/distinct_days:', order_range)
fx_dates


order_date min/max/distinct_days: (datetime.datetime(2024, 1, 16, 14, 30), datetime.datetime(2024, 12, 31, 15, 55), 236)


,date
0,2024-06-01
1,2024-09-15


## 13. Orders before the first fx_rates date

In [14]:
con.execute(f'''
    select count(*) as orders_before_first_fx_date
    from {orders}
    where order_date::date < date '2024-06-01'
''').df()


,orders_before_first_fx_date
0,55


## 14. fx_rates.json structure (responses flattened, error_example excluded)

In [15]:
con.execute(f'''
    select r.base, r.date, r.rates
    from (select unnest(responses) as r from {fx_rates})
    order by 1, 2
''').df()


,base,date,rates
0,EUR,2024-06-01,"{'USD': 1.077, 'EUR': 1.0, 'GBP': 0.852}"
1,EUR,2024-09-15,"{'USD': 1.1059, 'EUR': 1.0, 'GBP': 0.8436}"
2,GBP,2024-06-01,"{'USD': 1.2641, 'EUR': 1.1737, 'GBP': 1.0}"
3,USD,2024-06-01,"{'USD': 1.0, 'EUR': 0.9285, 'GBP': 0.7911}"
4,USD,2024-09-15,"{'USD': 1.0, 'EUR': 0.9042, 'GBP': 0.7628}"


## 15. products.json

In [16]:
con.execute(f'''
    select count(*) as rows, count(distinct id) as distinct_ids, count(distinct category) as categories
    from {products}
''').df()


,rows,distinct_ids,categories
0,100,100,10


## 16. Product ranking sensitivity: forward-fill pre-June orders in / out

In [17]:
con.execute(f'''
create temp table fx_flat as
select r.base, r.date,
    r.rates.USD as rate_usd, r.rates.EUR as rate_eur, r.rates.GBP as rate_gbp
from (select unnest(responses) as r from {fx_rates})
''')

con.execute("create temp table fx_dates as select distinct date from fx_flat order by 1")

con.execute(f'''
create temp table item_base as
select
    i.id as item_id, i.order_id, i.product_id, i.quantity, i.unit_price,
    i.currency as item_currency, o.order_date::date as order_day
from {order_items} i join {orders} o on i.order_id = o.id
''')

con.execute('''
create temp table item_fx_date as
select b.*,
    coalesce(
        (select max(d.date) from fx_dates d where d.date <= b.order_day),
        (select min(d.date) from fx_dates d)
    ) as fx_date
from item_base b
''')

con.execute('''
create temp table item_rate as
select f.*,
    case
        when f.item_currency = 'USD' then 1.0
        when direct.base in ('EUR', 'GBP') then direct.rate_usd
        when f.item_currency = 'EUR' and inv.rate_eur is not null then 1.0 / inv.rate_eur
        when f.item_currency = 'GBP' and inv.rate_gbp is not null then 1.0 / inv.rate_gbp
        else null
    end as rate_to_usd
from item_fx_date f
left join fx_flat direct on direct.base = f.item_currency and direct.date = f.fx_date
left join fx_flat inv on inv.base = 'USD' and inv.date = f.fx_date
''')

con.execute(f'''
create temp table item_revenue as
select r.*, r.quantity * r.unit_price * r.rate_to_usd as revenue_usd,
    coalesce(p.name, 'Unknown') as product_name
from item_rate r left join {products} p on r.product_id = p.id
''')

top_all = con.execute('''
    select product_id, product_name, round(sum(revenue_usd), 2) as revenue_usd
    from item_revenue where revenue_usd is not null
    group by 1, 2 order by 3 desc limit 10
''').df()

top_excl = con.execute('''
    select product_id, product_name, round(sum(revenue_usd), 2) as revenue_usd
    from item_revenue where revenue_usd is not null and order_day >= date '2024-06-01'
    group by 1, 2 order by 3 desc limit 10
''').df()

print('top 10, all orders (pre-June forward-filled):')
print(top_all.to_string(index=False))
print()
print('top 10, excluding pre-June orders:')
print(top_excl.to_string(index=False))
print()
print('dropped from top 10:', set(top_all['product_id']) - set(top_excl['product_id']))
print('added to top 10:', set(top_excl['product_id']) - set(top_all['product_id']))


top 10, all orders (pre-June forward-filled):
 product_id                  product_name  revenue_usd
          9              USB-C Hub 7-in-1      1693.60
         15           Leather Ankle Boots      1477.88
          8       Tablet Stand Adjustable      1464.17
         11        Organic Cotton T-Shirt      1390.55
         22    Mindfulness and Meditation      1382.01
         23      Cooking Around the World      1370.09
         37    Microfiber Cleaning Cloths      1369.23
         29 Machine Learning Fundamentals      1346.00
         35            Memory Foam Pillow      1329.33
         53         Hair Styling Tool Set      1293.05

top 10, excluding pre-June orders:
 product_id                  product_name  revenue_usd
         37    Microfiber Cleaning Cloths      1369.23
         35            Memory Foam Pillow      1329.33
         22    Mindfulness and Meditation      1304.70
         53         Hair Styling Tool Set      1293.05
         75           Dashboard Camera